# **Dynamic Bioacoustic State-Space Analysis of Eurasian Blue Tit Song**

### Latent Acoustic Geometry, Motif Grammar, and Temporal Bioacoustic Topology

Name: Sabrina Palis

Repository: https://github.com/MinervaRose/bioacoustic-topology

This notebook explores whether birdsong can be modeled as a structured dynamic system rather than as isolated acoustic events.

Using manifold learning, temporal trajectory modeling, probabilistic motif transitions, and acoustic novelty detection, the framework constructs a latent bioacoustic state-space representation of vocal behavior.

The system integrates:
- nonlinear acoustic embedding,
- temporal sequencing,
- motif recurrence analysis,
- anomaly detection,
- and interactive audio-synchronized visualization.

The objective is not to claim semantic language decoding, but rather to investigate whether vocal behavior exhibits measurable structural regularities analogous to computational syntax, attractor dynamics, or recurrent state transitions.

This notebook is intended as an exploratory computational bioacoustics research prototype.

---

## Research Questions

1. Can birdsong syllables be embedded into coherent latent acoustic manifolds?
2. Do recurrent vocal motifs exhibit structured transition probabilities?
3. Are there measurable acoustic attractor states?
4. Can anomalous vocal events be computationally identified?
5. Does birdsong exhibit recurrent temporal trajectory structure?

---

## Important Limitation

This framework does **not** claim:
- semantic translation,
- linguistic equivalence,
- or direct decoding of bird cognition.

All outputs should be interpreted as computational approximations of acoustic structure rather than proof of symbolic language.

# Methodology Overview

The computational pipeline follows the following stages:

1. Audio acquisition and metadata extraction
2. Syllable segmentation
3. Acoustic feature extraction
4. Latent manifold embedding
5. Clustering of vocal motifs
6. Temporal trajectory modeling
7. Motif transition probability estimation
8. Acoustic anomaly detection
9. Interactive audio-synchronized visualization

The workflow combines methods from:
- computational bioacoustics,
- nonlinear manifold learning,
- dynamical systems analysis,
- and probabilistic sequence modeling.

In [ ]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================

!pip install librosa soundfile plotly umap-learn requests pandas scikit-learn -q

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import requests
import numpy as np
import pandas as pd

import librosa
import librosa.display

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

import umap

import soundfile as sf

# Secure API Authentication

To avoid exposing credentials inside the notebook or GitHub repository, API authentication is handled through Google Colab Secrets.

The required secret is:

`XENO_CANTO_API_KEY`

This approach improves:
- reproducibility,
- security,
- and collaborative portability.

## Colab Setup

In Google Colab:

1. Open the 🔑 Secrets panel
2. Add a new secret:
   - Name: `XENO_CANTO_API_KEY`
   - Value: your API key
3. Enable notebook access

In [ ]:
# ============================================================
# SEARCH XENO CANTO
# ============================================================


# ============================================================
# LOAD XENO-CANTO API KEY FROM COLAB SECRETS
# ============================================================

from google.colab import userdata

API_KEY = userdata.get("XENO_CANTO_API_KEY")

if API_KEY is None:
    raise ValueError(
        "Missing XENO_CANTO_API_KEY in Colab Secrets."
    )

print("Xeno-canto API key loaded successfully.")

# ============================================================
# QUERY XENO-CANTO RECORDINGS
# ============================================================

url = "https://xeno-canto.org/api/3/recordings"

params = {
    "query": 'sp:"Cyanistes caeruleus" type:song q:A',
    "key": API_KEY
}

response = requests.get(url, params=params)

if response.status_code != 200:
    raise ValueError(
        f"API request failed with status "
        f"{response.status_code}"
    )

data = response.json()

records = pd.DataFrame(data["recordings"])

cols = [
    "id",
    "en",
    "gen",
    "sp",
    "cnt",
    "loc",
    "date",
    "length",
    "type",
    "q",
    "file"
]

records[cols].head(10)

In [ ]:
# ============================================================
# SELECT ONE RECORDING
# ============================================================

record = records.iloc[0]

audio_url = record["file"]

print(audio_url)

In [ ]:
# ============================================================
# RECORDING METADATA / PROVENANCE
# ============================================================

recording_metadata = {
    "xeno_canto_id": record.get("id", ""),
    "catalog_number": record.get("url", ""),
    "common_name": record.get("en", ""),
    "scientific_name": f"{record.get('gen', '')} {record.get('sp', '')}",
    "recordist": record.get("rec", ""),
    "country": record.get("cnt", ""),
    "location": record.get("loc", ""),
    "date": record.get("date", ""),
    "time": record.get("time", ""),
    "quality": record.get("q", ""),
    "type": record.get("type", ""),
    "length": record.get("length", ""),
    "audio_url": audio_url
}

recording_metadata

In [ ]:
# ============================================================
# DOWNLOAD AUDIO
# ============================================================

audio_path = "blue_tit.mp3"

audio_data = requests.get(audio_url).content

with open(audio_path, "wb") as f:
    f.write(audio_data)

print("Downloaded:", audio_path)

In [ ]:
# ============================================================
# LOAD AUDIO
# ============================================================

y, sr = librosa.load(audio_path)

print("Sample rate:", sr)
print("Audio length (seconds):", len(y)/sr)

In [ ]:
# ============================================================
# PLAY AUDIO
# ============================================================

from IPython.display import Audio

Audio(y, rate=sr)


In [ ]:
# ============================================================
# ONSET DETECTION
# ============================================================


onset_frames = librosa.onset.onset_detect(
    y=y,
    sr=sr,
    hop_length=512,
    backtrack=True,
    pre_max=20,
    post_max=20,
    pre_avg=100,
    post_avg=100,
    delta=0.2,
    wait=10
)

onset_times = librosa.frames_to_time(
    onset_frames,
    sr=sr
)

print("Detected onsets:", len(onset_times))

In [ ]:
# ============================================================
# BUILD ONSET INTERVALS
# ============================================================


intervals = []

for i in range(len(onset_times)-1):

    start = onset_times[i]
    end = onset_times[i+1]

    duration = end - start

    # skip tiny fragments
    if duration < 0.04:
        continue

    intervals.append([
        int(start * sr),
        int(end * sr)
    ])

intervals = np.array(intervals)

print("Intervals:", len(intervals))

# Convert to seconds for visualization
intervals_seconds = intervals / sr

In [ ]:
# ============================================================
# WAVEFORM
# ============================================================

plt.figure(figsize=(16,4))

librosa.display.waveshow(y, sr=sr)

plt.title("Blue Tit Song Waveform")
plt.xlabel("Time")
plt.ylabel("Amplitude")

plt.show()

In [ ]:
# ============================================================
# SPECTROGRAM
# ============================================================

D = librosa.amplitude_to_db(
    np.abs(librosa.stft(y)),
    ref=np.max
)

plt.figure(figsize=(16,6))

librosa.display.specshow(
    D,
    sr=sr,
    x_axis='time',
    y_axis='log'
)

plt.colorbar(format='%+2.0f dB')

plt.title("Blue Tit Song Spectrogram")

plt.show()

In [ ]:
# ============================================================
# VISUALIZE DETECTED SYLLABLES ON WAVEFORM
# ============================================================

plt.figure(figsize=(16, 4))
librosa.display.waveshow(y, sr=sr)

for start, end in intervals_seconds:
    plt.axvspan(start, end, alpha=0.25)

plt.title("Detected Blue Tit Syllables / Vocal Events")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.show()


# Acoustic Feature Extraction

Each syllable is transformed into a multidimensional acoustic descriptor vector.

Features include:
- MFCC-derived structure,
- spectral centroid,
- spectral bandwidth,
- temporal duration,
- and additional spectral statistics.

These descriptors provide a compact representation of local acoustic morphology.

MFCC-based representations were selected because they are widely used in:
- speech analysis,
- animal vocalization research,
- and auditory pattern recognition systems.

---

## Methodological Note

The extracted features do not directly encode meaning or behavior.

Rather, they provide a computational approximation of acoustic similarity relationships between vocal events.

In [ ]:
# ============================================================
# SYLLABLE-LEVEL FEATURE EXTRACTION
# ============================================================

syllable_features = []
syllable_metadata = []

for idx, (start_sample, end_sample) in enumerate(intervals):

    segment = y[start_sample:end_sample]

    # Skip very tiny fragments
    duration = len(segment) / sr
    if duration < 0.05:
        continue

    mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
    centroid = librosa.feature.spectral_centroid(y=segment, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=segment, sr=sr)
    rms_seg = librosa.feature.rms(y=segment)
    zcr = librosa.feature.zero_crossing_rate(segment)

    # pitch extraction
    f0, voiced_flag, voiced_probs = librosa.pyin(
    segment,
    fmin=librosa.note_to_hz("C3"),
    fmax=librosa.note_to_hz("C7")
    )

    f0_mean = (
    np.nanmean(f0)
    if np.any(~np.isnan(f0))
    else 0
    )

    f0_std = (
    np.nanstd(f0)
    if np.any(~np.isnan(f0))
    else 0
    )

    feature_vector = np.hstack([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        centroid.mean(),
        centroid.std(),
        bandwidth.mean(),
        bandwidth.std(),
        rms_seg.mean(),
        rms_seg.std(),
        zcr.mean(),
        zcr.std(),
        f0_mean,
        f0_std,
        duration
    ])

    syllable_features.append(feature_vector)

    syllable_metadata.append({
        "syllable_id": idx,
        "start_time": start_sample / sr,
        "end_time": end_sample / sr,
        "duration": duration
    })

syllable_features = np.array(syllable_features)
syllable_df = pd.DataFrame(syllable_metadata)

print("Syllable feature matrix:", syllable_features.shape)
syllable_df.head()

In [ ]:
# ============================================================
# NORMALIZE SYLLABLE FEATURES
# ============================================================

scaler = StandardScaler()
syllable_features_scaled = scaler.fit_transform(syllable_features)

# Latent Acoustic State-Space Embedding

The high-dimensional acoustic feature space is projected into a lower-dimensional latent manifold using UMAP (Uniform Manifold Approximation and Projection).

UMAP was selected because:
- it preserves local neighborhood topology,
- captures nonlinear structure,
- and performs well in heterogeneous acoustic datasets.

Within the resulting latent space:
- each node corresponds to a vocal syllable,
- spatial proximity approximates acoustic similarity,
- and trajectory movement approximates temporal vocal evolution.

This latent space should not be interpreted as a literal biological neural representation. It is instead a computational abstraction designed for exploratory structural analysis.

In [ ]:
# ============================================================
# UMAP SYLLABLE EMBEDDINGS
# ============================================================

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=5,
    min_dist=0.35,
    spread=1.8,
    metric="cosine",
    random_state=42
)

syllable_embedding = reducer.fit_transform(
    syllable_features_scaled
)

syllable_df["x"] = syllable_embedding[:,0]
syllable_df["y"] = syllable_embedding[:,1]
syllable_df["z"] = syllable_embedding[:,2]

syllable_df["time_index"] = np.arange(len(syllable_df))

syllable_df.head()

# Vocal Motif State Discovery

Clustering is used to identify recurrent acoustic states within the latent manifold.

These clusters are interpreted as:
- candidate vocal motifs,
- acoustic attractor regions,
- or recurrent dynamical states.

Importantly, cluster assignments are computational constructs rather than biologically validated vocal categories.

The objective is exploratory:
to determine whether recurring structural organization emerges from the acoustic manifold.

In [ ]:
# ============================================================
# CLUSTER SYLLABLE MOTIFS USING K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

clusters = kmeans.fit_predict(
    syllable_embedding
)

syllable_df["cluster"] = clusters

In [ ]:
# ============================================================
# LATENT TRAJECTORY DYNAMICS
# ============================================================

coords = syllable_df[["x", "y", "z"]].values

velocity = np.zeros(len(coords))
acceleration = np.zeros(len(coords))

for i in range(1, len(coords)):
    velocity[i] = np.linalg.norm(coords[i] - coords[i-1])

for i in range(2, len(coords)):
    acceleration[i] = abs(velocity[i] - velocity[i-1])

syllable_df["velocity"] = velocity
syllable_df["acceleration"] = acceleration

syllable_df[
    [
        "syllable_id",
        "start_time",
        "duration",
        "cluster",
        "velocity",
        "acceleration"
    ]
].head()

# Acoustic Novelty Detection

Novel vocal events are identified using trajectory-based anomaly scoring.

The objective is to computationally identify:
- unusual acoustic events,
- rare motif transitions,
- or potential exploratory vocal behaviors.

Novelty detection is useful because biological communication systems often contain:
- rare signaling events,
- adaptive improvisations,
- or context-dependent vocal deviations.

These detections are exploratory and require biological validation.

In [ ]:
# ============================================================
# ACOUSTIC NOVELTY / ANOMALY SCORE
# ============================================================

from sklearn.ensemble import IsolationForest

anomaly_model = IsolationForest(
    contamination=0.12,
    random_state=42
)

anomaly_labels = anomaly_model.fit_predict(syllable_features_scaled)

# Higher score = more unusual
anomaly_score = -anomaly_model.score_samples(syllable_features_scaled)

syllable_df["anomaly_score"] = anomaly_score
syllable_df["is_anomaly"] = anomaly_labels == -1

syllable_df[
    [
        "syllable_id",
        "start_time",
        "cluster",
        "velocity",
        "acceleration",
        "anomaly_score",
        "is_anomaly"
    ]
].sort_values("anomaly_score", ascending=False).head(10)

# Temporal Acoustic Trajectories

Birdsong is inherently sequential.

To model this temporal structure, syllables are connected according to their order of occurrence within the recording.

This transforms the latent manifold into a dynamic trajectory system in which:
- nodes represent syllables,
- edges represent temporal progression,
- and movement through state-space approximates vocal evolution over time.

The resulting trajectories enable analysis of:
- motif recurrence,
- directional transitions,
- state persistence,
- and dynamical vocal behavior.

In [ ]:
# ============================================================
# BUILD SOFT ACOUSTIC FIELD
# ============================================================

coords = syllable_df[["x","y","z"]].values

edge_x = []
edge_y = []
edge_z = []

# ------------------------------------------------------------
# LOCAL SIMILARITY CONNECTIONS
# ------------------------------------------------------------

for i in range(len(coords)):

    for j in range(i+1, len(coords)):

        dist = np.linalg.norm(
            coords[i] - coords[j]
        )

        same_cluster = (
            syllable_df["cluster"].iloc[i]
            ==
            syllable_df["cluster"].iloc[j]
        )

        # dense local motif field
        if dist < 2.5 and same_cluster:

            edge_x += [coords[i,0], coords[j,0], None]
            edge_y += [coords[i,1], coords[j,1], None]
            edge_z += [coords[i,2], coords[j,2], None]

# ------------------------------------------------------------
# TEMPORAL TRAJECTORIES
# ------------------------------------------------------------

for i in range(len(coords)-1):

    dist = np.linalg.norm(
        coords[i] - coords[i+1]
    )

    if dist < 5:

        edge_x += [coords[i,0], coords[i+1,0], None]
        edge_y += [coords[i,1], coords[i+1,1], None]
        edge_z += [coords[i,2], coords[i+1,2], None]

print("Soft acoustic field built.")

# Probabilistic Motif Transition Modeling

Transition probabilities are estimated between discovered motif states.

This creates a first-order probabilistic vocal grammar model.

The resulting transition matrix enables:
- identification of recurrent motif chains,
- estimation of state persistence,
- and detection of dominant sequential pathways.

High transition probabilities may indicate stable recurring vocal syntax-like structures.

However, this should not be interpreted as proof of linguistic grammar.

In [ ]:
# ============================================================
# MOTIF TRANSITION MATRIX
# Which motif group tends to follow which?
# ============================================================

transition_counts = pd.crosstab(
    syllable_df["cluster"].iloc[:-1].values,
    syllable_df["cluster"].iloc[1:].values,
    rownames=["from_cluster"],
    colnames=["to_cluster"]
)

transition_probs = transition_counts.div(
    transition_counts.sum(axis=1),
    axis=0
).fillna(0)

transition_counts

In [ ]:
# ============================================================
# HUMAN-READABLE MOTIF STATE LABELS
# ============================================================

cluster_labels = {
    0: "TERMINAL DESCENT",
    1: "VERTICAL BURST",
    2: "TRANSITION CLOUD",
    3: "STABLE CHIRP FIELD"
}

syllable_df["motif_label"] = syllable_df["cluster"].map(cluster_labels)

syllable_df[
    [
        "syllable_id",
        "cluster",
        "motif_label",
        "velocity",
        "acceleration",
        "anomaly_score"
    ]
].head()

In [ ]:
# ============================================================
# LABELED MOTIF TRANSITION MATRIX
# ============================================================

labeled_transition_probs = transition_probs.copy()

labeled_transition_probs.index = [
    cluster_labels.get(i, f"Cluster {i}")
    for i in labeled_transition_probs.index
]

labeled_transition_probs.columns = [
    cluster_labels.get(i, f"Cluster {i}")
    for i in labeled_transition_probs.columns
]

labeled_transition_probs

In [ ]:
# ============================================================
# LABELED MOTIF TRANSITION HEATMAP
# ============================================================

fig = px.imshow(
    labeled_transition_probs,
    text_auto=".2f",
    color_continuous_scale="Turbo",
    title="BLUE TIT SONG // LABELED MOTIF TRANSITION MATRIX"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="#00ffff"),
    xaxis_title="To motif state",
    yaxis_title="From motif state"
)

fig.show()

In [ ]:
# ============================================================
# INTERPRET MOTIF TRANSITIONS
# ============================================================

print("Dominant motif transitions:")

for from_state in labeled_transition_probs.index:
    top_to_state = labeled_transition_probs.loc[from_state].idxmax()
    prob = labeled_transition_probs.loc[from_state].max()

    print(f"- {from_state} → {top_to_state}: {prob:.2f}")

In [ ]:
# ============================================================
# BIOACOUSTIC GRAMMAR REPORT
# ============================================================

self_loops = np.diag(labeled_transition_probs.values)

most_stable_idx = np.argmax(self_loops)
most_stable_state = labeled_transition_probs.index[most_stable_idx]
most_stable_score = self_loops[most_stable_idx]

strongest_transition = (
    labeled_transition_probs
    .where(~np.eye(len(labeled_transition_probs), dtype=bool))
    .stack()
    .idxmax()
)

strongest_transition_score = (
    labeled_transition_probs
    .where(~np.eye(len(labeled_transition_probs), dtype=bool))
    .stack()
    .max()
)

print("BIOACOUSTIC GRAMMAR REPORT")
print("=" * 36)
print(f"Most stable motif state: {most_stable_state} ({most_stable_score:.2f})")
print(
    f"Strongest cross-state transition: "
    f"{strongest_transition[0]} → {strongest_transition[1]} "
    f"({strongest_transition_score:.2f})"
)
print()
print("Interpretation:")
print(
    f"- {most_stable_state} behaves like an acoustic attractor: "
    "the bird tends to remain in or return to this state."
)
print(
    f"- The transition {strongest_transition[0]} → {strongest_transition[1]} "
    "may represent a recurring phrase movement."
)
print(
    "- High transition probabilities suggest repeated syntax-like structure, "
    "not random vocal movement."
)

In [ ]:
# ============================================================
# MOTIF PHRASE RECURRENCE DETECTION
# Detect repeated 3-motif sequences
# ============================================================

from collections import Counter

motif_sequence = syllable_df["motif_label"].tolist()

phrase_length = 3

phrases = [
    tuple(motif_sequence[i:i+phrase_length])
    for i in range(len(motif_sequence) - phrase_length + 1)
]

phrase_counts = Counter(phrases)

recurrent_phrases = {
    phrase: count
    for phrase, count in phrase_counts.items()
    if count > 1
}

print("RECURRENT MOTIF PHRASES")
print("=" * 32)

for phrase, count in sorted(
    recurrent_phrases.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{' → '.join(phrase)} : {count} times")

In [ ]:
# ============================================================
# VISUALIZE RECURRENT MOTIF PHRASES
# ============================================================

phrase_df = pd.DataFrame([
    {
        "phrase": " → ".join(phrase),
        "count": count
    }
    for phrase, count in recurrent_phrases.items()
]).sort_values("count", ascending=True)

fig = px.bar(
    phrase_df,
    x="count",
    y="phrase",
    orientation="h",
    title="BLUE TIT SONG // RECURRENT MOTIF PHRASES",
    color="count",
    color_continuous_scale="Turbo"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="#00ffff"),
    xaxis_title="Recurrence count",
    yaxis_title="Motif phrase"
)

fig.show()

In [ ]:
# ============================================================
# MOTIF PHRASE INTELLIGENCE SUMMARY
# ============================================================

top_phrase = max(recurrent_phrases.items(), key=lambda x: x[1])
top_phrase_text = " → ".join(top_phrase[0])
top_phrase_count = top_phrase[1]

num_recurrent_phrases = len(recurrent_phrases)

print("MOTIF PHRASE INTELLIGENCE SUMMARY")
print("=" * 42)
print(f"Most recurrent phrase: {top_phrase_text}")
print(f"Occurrences: {top_phrase_count}")
print(f"Number of recurrent phrase types: {num_recurrent_phrases}")
print()
print("Interpretation:")
print(
    "- The recording contains repeated motif chains, suggesting structured vocal sequencing."
)
print(
    f"- The dominant phrase pattern is: {top_phrase_text}."
)
print(
    "- This can be treated as a candidate acoustic syntax fragment for further comparison across recordings."
)

In [ ]:
from google.colab import files
files.download("blue_tit_syllable_intelligence_report.csv")

In [ ]:
# Final static topology visualization
# ============================================================
#  BIOACOUSTIC TOPOLOGY WITH TRAJECTORY DYNAMICS
# Node size = duration
# Node color = velocity
# Halo size = acceleration
# ============================================================

import plotly.graph_objects as go

sizes = 6 + 35 * syllable_df["duration"] / syllable_df["duration"].max()

halo_sizes = 8 + 60 * syllable_df["acceleration"] / max(
    syllable_df["acceleration"].max(),
    1e-9
)

# ------------------------------------------------------------
# MAIN NODES: velocity-coded
# ------------------------------------------------------------

node_trace = go.Scatter3d(
    x=syllable_df["x"],
    y=syllable_df["y"],
    z=syllable_df["z"],
    mode="markers",
    marker=dict(
        size=sizes,
        color=syllable_df["velocity"],
        colorscale="Turbo",
        opacity=0.95,
        colorbar=dict(title="Velocity"),
        line=dict(width=0)
    ),
    text=[
        f"Syllable {row.syllable_id}<br>"
        f"Start: {row.start_time:.2f}s<br>"
        f"Duration: {row.duration:.2f}s<br>"
        f"Cluster: {row.cluster}<br>"
        f"Motif: {row.motif_label}<br>"
        f"Velocity: {row.velocity:.2f}<br>"
        f"Acceleration: {row.acceleration:.2f}<br>"
        f"Novelty: {row.anomaly_score:.3f}<br>"
        f"Novel event: {row.is_anomaly}"
        for row in syllable_df.itertuples()
    ],
    hoverinfo="text",
    name="syllables"
)

# ------------------------------------------------------------
# NOVEL EVENT ALERT HALOS
# ------------------------------------------------------------

novel_df = syllable_df[syllable_df["is_anomaly"]]

novel_halo_trace = go.Scatter3d(
    x=novel_df["x"],
    y=novel_df["y"],
    z=novel_df["z"],
    mode="markers",
    marker=dict(
        size=28,
        color="red",
        opacity=0.12,
        line=dict(width=0)
    ),
    hoverinfo="none",
    showlegend=False
)

# ------------------------------------------------------------
# NOVEL ACOUSTIC EVENTS
# ------------------------------------------------------------



novel_trace = go.Scatter3d(
    x=novel_df["x"],
    y=novel_df["y"],
    z=novel_df["z"],
    mode="markers",
    marker=dict(
        size=9,
        color="red",
        symbol="diamond",
        opacity=0.95,
        line=dict(width=1, color="rgba(255,255,255,0.8)")
    ),
    text=[
        f"NOVEL EVENT<br>"
        f"Syllable {row.syllable_id}<br>"
        f"Start: {row.start_time:.2f}s<br>"
        f"Novelty: {row.anomaly_score:.3f}<br>"
        f"Velocity: {row.velocity:.2f}<br>"
        f"Acceleration: {row.acceleration:.2f}"
        for row in novel_df.itertuples()
    ],
    hoverinfo="text",
    name="novel acoustic events"
)

# ------------------------------------------------------------
# HALO LAYER: acceleration-coded
# ------------------------------------------------------------

halo_trace = go.Scatter3d(
    x=syllable_df["x"],
    y=syllable_df["y"],
    z=syllable_df["z"],
    mode="markers",
    marker=dict(
        size=halo_sizes,
        color=syllable_df["acceleration"],
        colorscale="Hot",
        opacity=0.13,
        line=dict(width=0)
    ),
    hoverinfo="none",
    showlegend=False
)

# ------------------------------------------------------------
# ACOUSTIC GRAPH EDGES
# Uses your existing edge_x, edge_y, edge_z
# ------------------------------------------------------------

edge_trace = go.Scatter3d(
    x=edge_x,
    y=edge_y,
    z=edge_z,
    mode="lines",
    line=dict(
        width=1.0,
        color="rgba(0,255,255,0.12)"
    ),
    hoverinfo="none",
    name="acoustic links"
)

# ------------------------------------------------------------
# LOCAL TEMPORAL FLOW
# Only connects nearby consecutive syllables
# ------------------------------------------------------------

traj_x = []
traj_y = []
traj_z = []

coords = syllable_df[["x", "y", "z"]].values

for i in range(len(coords)-1):

    dist = np.linalg.norm(coords[i] - coords[i+1])

    if dist < 4:
        traj_x += [coords[i,0], coords[i+1,0], None]
        traj_y += [coords[i,1], coords[i+1,1], None]
        traj_z += [coords[i,2], coords[i+1,2], None]

trajectory_trace = go.Scatter3d(
    x=traj_x,
    y=traj_y,
    z=traj_z,
    mode="lines",
    line=dict(
        color="rgba(255,255,255,0.28)",
        width=4
    ),
    hoverinfo="none",
    showlegend=False
)

# ------------------------------------------------------------
# FINAL FIGURE
# ------------------------------------------------------------

fig = go.Figure(
    data=[
        edge_trace,
        halo_trace,
        trajectory_trace,
        node_trace,
        novel_halo_trace,
        novel_trace
    ]
)

fig.update_layout(
    title="BLUE TIT SONG // DYNAMIC BIOACOUSTIC STATE-SPACE",
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
    height=900,
    scene=dict(
        xaxis=dict(
            title="Latent Acoustic Axis X",
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.15)",
            zerolinecolor="rgba(0,255,255,0.25)"
        ),
        yaxis=dict(
            title="Latent Acoustic Axis Y",
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.15)",
            zerolinecolor="rgba(0,255,255,0.25)"
        ),
        zaxis=dict(
            title="Latent Acoustic Axis Z",
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.15)",
            zerolinecolor="rgba(0,255,255,0.25)"
        ),
        camera=dict(
            eye=dict(x=0.8, y=2.2, z=1.0)
        )
    ),
    margin=dict(l=0, r=0, b=0, t=60)
)

# ------------------------------------------------------------
# INTELLIGENCE DASHBOARD ANNOTATION
# ------------------------------------------------------------

num_syllables = len(syllable_df)
num_clusters = syllable_df["cluster"].nunique()
num_novel = syllable_df["is_anomaly"].sum()
max_velocity = syllable_df["velocity"].max()
max_acceleration = syllable_df["acceleration"].max()

fig.add_annotation(
    text=(
        f"ACOUSTIC INTELLIGENCE SUMMARY<br>"
        f"Syllables tracked: {num_syllables}<br>"
        f"Motif groups: {num_clusters}<br>"
        f"Novel events: {num_novel}<br>"
        f"Max velocity: {max_velocity:.2f}<br>"
        f"Max acceleration: {max_acceleration:.2f}"
    ),
    x=0.02,
    y=0.98,
    xref="paper",
    yref="paper",
    showarrow=False,
    align="left",
    font=dict(
        size=13,
        color="#00ffff",
        family="monospace"
    ),
    bgcolor="rgba(0,0,0,0.55)",
    bordercolor="rgba(0,255,255,0.45)",
    borderwidth=1
)

fig.show()

# Interactive Audio-Synchronized Bioacoustic Visualization

The final stage integrates:
- latent state-space visualization,
- temporal trajectories,
- anomaly highlighting,
- metadata overlays,
- and synchronized audio playback.

The resulting interface functions as a computational bioacoustic observatory:
a dynamic environment for exploring vocal topology in real time.

The visualization is designed to support:
- exploratory analysis,
- pattern discovery,
- and future comparative bioacoustic research.

In [ ]:
# ============================================================
# TRUE AUDIO-SYNCED HTML EXPORT
# Audio plays while nodes appear at real syllable times
# ============================================================

import json
import base64
from pathlib import Path

df = syllable_df.sort_values("start_time").reset_index(drop=True)

# Encode audio as base64
audio_bytes = Path(audio_path).read_bytes()
audio_b64 = base64.b64encode(audio_bytes).decode("utf-8")

points = df[[
    "x", "y", "z",
    "start_time",
    "duration",
    "cluster",
    "motif_label",
    "syllable_id",
    "velocity",
    "acceleration",
    "anomaly_score",
    "is_anomaly"
]].to_dict(orient="records")

html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Blue Tit Bioacoustic Topology</title>
    <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
    <style>
        body {{
            margin: 0;
            background: black;
            color: #00ffff;
            font-family: monospace;
        }}
        #plot {{
            width: 100vw;
            height: 84vh;
        }}
        #controls {{
            padding: 12px;
            background: #020202;
            border-top: 1px solid rgba(0,255,255,0.3);
        }}
        #metadata {{
            margin-top: 8px;
            color: rgba(0,255,255,0.75);
            font-size: 12px;
            letter-spacing: 1px;
        }}
        button {{
            background: black;
            color: #00ffff;
            border: 1px solid #00ffff;
            padding: 8px 16px;
            font-family: monospace;
            cursor: pointer;
        }}
        #hudPanel {{
            position: absolute;
            top: 70px;
            left: 24px;
            padding: 14px 18px;
            color: #00ffff;
            background: rgba(0, 20, 25, 0.55);
            border: 1px solid rgba(0,255,255,0.55);
            box-shadow:
                0 0 12px rgba(0,255,255,0.45),
                inset 0 0 18px rgba(0,255,255,0.12);
            font-size: 13px;
            letter-spacing: 1px;
            z-index: 10;
        }}

        .hudTitle {{
            font-weight: bold;
            margin-bottom: 8px;
            color: white;
            text-shadow: 0 0 8px cyan;
        }}

        #legendPanel {{
            position: absolute;
            top: 70px;
            right: 24px;
            padding: 14px 18px;
            color: #00ffff;
            background: rgba(0, 20, 25, 0.55);
            border: 1px solid rgba(0,255,255,0.55);
            box-shadow:
                0 0 12px rgba(0,255,255,0.45),
                inset 0 0 18px rgba(0,255,255,0.12);
            font-size: 13px;
            letter-spacing: 1px;
            z-index: 10;
        }}

        .dot.normal {{
            display: inline-block;
            width: 10px;
            height: 10px;
            background: cyan;
            border-radius: 50%;
            margin-right: 8px;
        }}

        .diamond {{
            display: inline-block;
            width: 11px;
            height: 11px;
            background: red;
            transform: rotate(45deg);
            margin-right: 8px;
        }}

    </style>
</head>

<body>

<div id="plot"></div>

<div id="hudPanel">
    <div class="hudTitle">BIOACOUSTIC INTELLIGENCE</div>
    <div>Syllables: {len(syllable_df)}</div>
    <div>Motif groups: {syllable_df["cluster"].nunique()}</div>
    <div>Novel events: {syllable_df["is_anomaly"].sum()}</div>
    <div>Max velocity: {syllable_df["velocity"].max():.2f}</div>
    <div>Max acceleration: {syllable_df["acceleration"].max():.2f}</div>
</div>

<div id="legendPanel">
    <div class="hudTitle">VISUAL LEGEND</div>
    <div><span class="dot normal"></span> Normal syllable</div>
    <div><span class="diamond"></span> Novel acoustic event</div>
     <div>Color: time progression</div>
    <div>Size: syllable duration</div>
</div>

<div id="controls">
    <button onclick="startExperience()">PLAY BLUE TIT TOPOLOGY</button>
    <span id="timeLabel"> t = 0.00s </span>
    <audio id="audio" controls>
        <source src="data:audio/mp3;base64,{audio_b64}" type="audio/mp3">
    </audio>

    <div id="metadata">
        XC-{recording_metadata["xeno_canto_id"]} //
        {recording_metadata["common_name"]} //
        {recording_metadata["scientific_name"]} //
        {recording_metadata["country"]} //
        {recording_metadata["date"]} //
        Quality {recording_metadata["quality"]}
    </div>
</div>

<script>
const points = {json.dumps(points)};

const xRange = [{df["x"].min()-1}, {df["x"].max()+1}];
const yRange = [{df["y"].min()-1}, {df["y"].max()+1}];
const zRange = [{df["z"].min()-1}, {df["z"].max()+1}];

let visibleX = [];
let visibleY = [];
let visibleZ = [];
let visibleColor = [];
let visibleSize = [];
let visibleText = [];

const maxDuration = Math.max(...points.map(p => p.duration));

let trace = {{
    x: visibleX,
    y: visibleY,
    z: visibleZ,
    mode: "markers",
    type: "scatter3d",
    marker: {{
        size: visibleSize,
        color: visibleColor,
        colorscale: "Turbo",
        opacity: 0.95
    }},
    text: visibleText,
    hoverinfo: "text"
}};

let layout = {{
    title: "BLUE TIT SONG // TRUE AUDIO-SYNCED BIOACOUSTIC TOPOLOGY",
    showlegend: false,
    paper_bgcolor: "black",
    plot_bgcolor: "black",
    font: {{ color: "#00ffff" }},
    scene: {{
        xaxis: {{
            title: "Latent Axis X",
            range: xRange,
            backgroundcolor: "black",
            gridcolor: "rgba(0,255,255,0.15)"
        }},
        yaxis: {{
            title: "Latent Axis Y",
            range: yRange,
            backgroundcolor: "black",
            gridcolor: "rgba(0,255,255,0.15)"
        }},
        zaxis: {{
            title: "Latent Axis Z",
            range: zRange,
            backgroundcolor: "black",
            gridcolor: "rgba(0,255,255,0.15)"
        }},
        camera: {{
            eye: {{x: 0.8, y: 2.2, z: 1.0}}
        }}
    }},
    margin: {{l: 0, r: 0, b: 0, t: 50}}
}};

Plotly.newPlot("plot", [trace], layout);

let nextPointIndex = 0;
let animationInterval = null;

function resetPlot() {{
    visibleX = [];
    visibleY = [];
    visibleZ = [];
    visibleColor = [];
    visibleSize = [];
    visibleText = [];
    nextPointIndex = 0;

    Plotly.react("plot", [{{
        ...trace,
        x: visibleX,
        y: visibleY,
        z: visibleZ,
        marker: {{
            ...trace.marker,
            size: visibleSize,
            color: visibleColor
        }},
        text: visibleText
    }}], layout);
}}

function updatePlot(currentTime) {{

    document.getElementById("timeLabel").innerText =
        " t = " + currentTime.toFixed(2) + "s ";

    let visible = points.filter(p => p.start_time <= currentTime);

    let normal = visible.filter(p => !p.is_anomaly);
    let novel = visible.filter(p => p.is_anomaly);

    function buildText(p) {{
        return "Syllable " + p.syllable_id +
            "<br>Start: " + p.start_time.toFixed(2) + "s" +
            "<br>Duration: " + p.duration.toFixed(2) + "s" +
            "<br>Motif: " + p.motif_label +
            "<br>Velocity: " + p.velocity.toFixed(2) +
            "<br>Acceleration: " + p.acceleration.toFixed(2) +
            "<br>Novelty: " + p.anomaly_score.toFixed(3);
    }}

    let normalTrace = {{
        type: "scatter3d",
        mode: "markers",
        x: normal.map(p => p.x),
        y: normal.map(p => p.y),
        z: normal.map(p => p.z),
        marker: {{
            size: normal.map(p => 8 + 35 * p.duration / maxDuration),
            color: normal.map(p => p.start_time),
            colorscale: "Turbo",
            opacity: 0.95
        }},
        text: normal.map(buildText),
        hoverinfo: "text",
        name: "normal syllables"
    }};

    let novelTrace = {{
        type: "scatter3d",
        mode: "markers",
        x: novel.map(p => p.x),
        y: novel.map(p => p.y),
        z: novel.map(p => p.z),
        marker: {{
            size: novel.map(p => 14 + 35 * p.duration / maxDuration),
            color: "red",
            symbol: "diamond",
            opacity: 0.95,
            line: {{
                width: 1,
                color: "white"
            }}
        }},
        text: novel.map(buildText),
        hoverinfo: "text",
        name: "novel acoustic events"
    }};

    Plotly.react("plot", [normalTrace, novelTrace], layout);
}}

function startExperience() {{

    const audio = document.getElementById("audio");

    audio.currentTime = 0;
    audio.play();

    if (animationInterval) {{
        clearInterval(animationInterval);
    }}

    animationInterval = setInterval(() => {{
        updatePlot(audio.currentTime);

        if (audio.ended) {{
            clearInterval(animationInterval);
        }}
    }}, 50);
}}
</script>

</body>
</html>
"""

output_path = "blue_tit_audio_synced_topology.html"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

print("Saved:", output_path)

In [ ]:
from google.colab import files
files.download("blue_tit_audio_synced_topology.html")

# Experimental Toroidal Vocal Geometry

## Cyclic State-Space Hypothesis

Linear embeddings may not fully capture recurrent or cyclic structure in biological vocal systems.

Many dynamical systems in nature evolve on:
- recurrent attractors,
- oscillatory loops,
- or cyclic manifolds.

To explore this possibility, the latent acoustic trajectories are additionally projected onto a toroidal geometry.

In this representation:
- angular dimensions represent cyclic vocal progression,
- repeated motif returns become geometrically visible,
- and recurrent vocal pathways may emerge as closed-loop structures.

This experiment is exploratory and should not be interpreted as evidence of literal toroidal neural organization.

Rather, it investigates whether cyclic manifold representations better capture recurrent birdsong dynamics than linear latent embeddings.

---

## Conceptual Motivation

If recurrent vocal motifs repeatedly return to similar acoustic states, a toroidal manifold may:
- preserve recurrence structure more naturally,
- reduce trajectory fragmentation,
- and reveal hidden cyclic organization.

This approach draws inspiration from:
- dynamical systems theory,
- recurrent state-space modeling,
- and nonlinear topology analysis.

In [ ]:
# ============================================================
# TOROIDAL VOCAL GEOMETRY
# Experimental cyclic embedding
# ============================================================

import numpy as np

df_torus = syllable_df.copy()

# ------------------------------------------------------------
# CYCLIC PARAMETERS
# ------------------------------------------------------------

theta = np.linspace(
    0,
    2 * np.pi,
    len(df_torus)
)

# motif-dependent minor angle
phi = (
    df_torus["cluster"]
    / df_torus["cluster"].max()
) * 2 * np.pi

# torus radii
R = 12
r = 4

# ------------------------------------------------------------
# TORUS COORDINATES
# ------------------------------------------------------------

x_torus = (R + r * np.cos(phi)) * np.cos(theta)
y_torus = (R + r * np.cos(phi)) * np.sin(theta)
z_torus = r * np.sin(phi)

df_torus["x_torus"] = x_torus
df_torus["y_torus"] = y_torus
df_torus["z_torus"] = z_torus

df_torus.head()

In [ ]:
# ============================================================
# TOROIDAL BIOACOUSTIC VISUALIZATION
# ============================================================

import plotly.graph_objects as go

sizes = 6 + 35 * (
    df_torus["duration"]
    / df_torus["duration"].max()
)

fig = go.Figure()

# ------------------------------------------------------------
# TORUS TRAJECTORY
# ------------------------------------------------------------

fig.add_trace(

    go.Scatter3d(

        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],

        mode="lines",

        line=dict(
            color="rgba(255,255,255,0.25)",
            width=4
        ),

        hoverinfo="none",
        showlegend=False
    )
)

# ------------------------------------------------------------
# TORUS NODES
# ------------------------------------------------------------

fig.add_trace(

    go.Scatter3d(

        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],

        mode="markers",

        marker=dict(
            size=sizes,
            color=df_torus["time_index"],
            colorscale="Turbo",
            opacity=0.95,
            line=dict(width=0)
        ),

        text=[
            f"Syllable {row.syllable_id}<br>"
            f"Motif: {row.motif_label}<br>"
            f"Velocity: {row.velocity:.2f}<br>"
            f"Novelty: {row.anomaly_score:.3f}"
            for row in df_torus.itertuples()
        ],

        hoverinfo="text",

        name="toroidal vocal states"
    )
)

# ------------------------------------------------------------
# NOVEL EVENT MARKERS
# ------------------------------------------------------------

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(

    go.Scatter3d(

        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],

        mode="markers",

        marker=dict(
            size=10,
            color="red",
            symbol="diamond",
            opacity=0.95,
            line=dict(width=1, color="white")
        ),

        hoverinfo="none",
        name="novel acoustic events"
    )
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig.update_layout(

    title="BLUE TIT SONG // TOROIDAL VOCAL GEOMETRY",

    template="plotly_dark",

    paper_bgcolor="black",
    plot_bgcolor="black",

    height=950,

    scene=dict(

        xaxis=dict(
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.12)"
        ),

        yaxis=dict(
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.12)"
        ),

        zaxis=dict(
            backgroundcolor="black",
            gridcolor="rgba(0,255,255,0.12)"
        ),

        camera=dict(
            eye=dict(x=1.5, y=1.8, z=0.9)
        )
    )
)

fig.show()

# Toroidal Trajectory Flow Dynamics

The toroidal embedding enables visualization of cyclic vocal circulation through latent acoustic state-space.

Directional trajectory flow may reveal:
- recurrent acoustic loops,
- motif orbit behavior,
- stable attractor circulation,
- and cyclic vocal revisitation patterns.

Rather than treating birdsong as isolated events, this representation models vocalization as movement through a continuous dynamical manifold.

In [ ]:
# ============================================================
# TOROIDAL FLOW VECTORS
# ============================================================

flow_x = []
flow_y = []
flow_z = []

for i in range(len(df_torus) - 1):

    x0 = df_torus.iloc[i]["x_torus"]
    y0 = df_torus.iloc[i]["y_torus"]
    z0 = df_torus.iloc[i]["z_torus"]

    x1 = df_torus.iloc[i + 1]["x_torus"]
    y1 = df_torus.iloc[i + 1]["y_torus"]
    z1 = df_torus.iloc[i + 1]["z_torus"]

    flow_x.extend([x0, x1, None])
    flow_y.extend([y0, y1, None])
    flow_z.extend([z0, z1, None])

fig = go.Figure()

# ------------------------------------------------------------
# FLOW TRAJECTORIES
# ------------------------------------------------------------

fig.add_trace(

    go.Scatter3d(

        x=flow_x,
        y=flow_y,
        z=flow_z,

        mode="lines",

        line=dict(
            color="rgba(0,255,255,0.55)",
            width=5
        ),

        hoverinfo="none",
        name="trajectory flow"
    )
)

# ------------------------------------------------------------
# NODES
# ------------------------------------------------------------

fig.add_trace(

    go.Scatter3d(

        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],

        mode="markers",

        marker=dict(
            size=7 + 30 * (
                df_torus["duration"]
                / df_torus["duration"].max()
            ),

            color=df_torus["time_index"],
            colorscale="Turbo",
            opacity=0.92
        ),

        text=df_torus["motif_label"],

        hoverinfo="text",
        name="vocal states"
    )
)

# ------------------------------------------------------------
# ANOMALIES
# ------------------------------------------------------------

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(

    go.Scatter3d(

        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],

        mode="markers",

        marker=dict(
            size=11,
            color="red",
            symbol="diamond"
        ),

        hoverinfo="none",
        name="novel events"
    )
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig.update_layout(

    title="BLUE TIT SONG // TOROIDAL FLOW DYNAMICS",

    template="plotly_dark",

    paper_bgcolor="black",
    plot_bgcolor="black",

    height=950,

    scene=dict(

        bgcolor="black",

        xaxis=dict(
            gridcolor="rgba(0,255,255,0.15)"
        ),

        yaxis=dict(
            gridcolor="rgba(0,255,255,0.15)"
        ),

        zaxis=dict(
            gridcolor="rgba(0,255,255,0.15)"
        ),

        camera=dict(
            eye=dict(x=1.7, y=1.8, z=1.2)
        )
    )
)

fig.show()

# Toroidal Topology Interpretation

The toroidal projection suggests that recurrent vocal motifs may occupy cyclic acoustic regions rather than purely linear latent trajectories.

Several observations emerge:

- motif groups appear spatially segregated along orbital regions,
- recurrent phrase transitions generate repeated circulation pathways,
- anomalous acoustic events frequently occur near high-curvature transitions,
- and stable vocal states appear associated with persistent orbital occupation.

This may indicate that vocal sequencing contains:
- recurrent dynamical structure,
- attractor-like state persistence,
- and cyclic revisitation behavior.

Importantly, these interpretations remain exploratory computational observations rather than biological conclusions.

In [ ]:
# ============================================================
# TOROIDAL REGION ANALYSIS
# ============================================================

print("TOROIDAL REGION ANALYSIS")
print("=" * 45)

for cluster_id in sorted(df_torus["cluster"].unique()):

    cluster_df = df_torus[
        df_torus["cluster"] == cluster_id
    ]

    mean_x = cluster_df["x_torus"].mean()
    mean_y = cluster_df["y_torus"].mean()
    mean_z = cluster_df["z_torus"].mean()

    spread = np.mean(
        np.sqrt(
            (cluster_df["x_torus"] - mean_x) ** 2 +
            (cluster_df["y_torus"] - mean_y) ** 2 +
            (cluster_df["z_torus"] - mean_z) ** 2
        )
    )

    print()
    print(f"Motif: {cluster_labels[cluster_id]}")
    print(f"Mean toroidal position:")
    print(f"  x = {mean_x:.2f}")
    print(f"  y = {mean_y:.2f}")
    print(f"  z = {mean_z:.2f}")
    print(f"Spatial spread: {spread:.2f}")

## Toroidal Region Analysis Interpretation

The toroidal region analysis suggests that the discovered motif states occupy different cyclic regions of the experimental vocal manifold.

The **VERTICAL BURST** and **TRANSITION CLOUD** states show lower spatial spread, suggesting more compact orbital regions.

The **TERMINAL DESCENT** and **STABLE CHIRP FIELD** states show larger spread, suggesting broader or more variable recurrent behavior across the toroidal projection.

This does not prove biological toroidal organization, but it supports the use of cyclic geometry as an exploratory tool for identifying recurrent vocal state structure.

# Holographic Acoustic Density Fields

To better visualize recurrent acoustic attractor regions, motif states are rendered as translucent volumetric density fields.

These holographic fields approximate:
- regions of persistent vocal occupation,
- motif concentration zones,
- and recurrent acoustic attractors.

Rather than emphasizing isolated nodes, the visualization treats vocal states as distributed energetic regions within latent acoustic topology.

This representation is inspired by:
- dynamical systems visualization,
- volumetric state-space rendering,
- and computational field modeling.

In [ ]:
# ============================================================
# HOLOGRAPHIC MOTIF DENSITY FIELDS
# ============================================================

fig = go.Figure()

cluster_colors = {
    0: "rgba(255,80,80,0.16)",
    1: "rgba(80,255,255,0.16)",
    2: "rgba(255,200,80,0.16)",
    3: "rgba(120,255,120,0.16)"
}

for cluster_id in sorted(df_torus["cluster"].unique()):

    cluster_df = df_torus[df_torus["cluster"] == cluster_id]

    fig.add_trace(
        go.Scatter3d(
            x=cluster_df["x_torus"],
            y=cluster_df["y_torus"],
            z=cluster_df["z_torus"],
            mode="markers",
            marker=dict(
                size=38,
                color=cluster_colors[cluster_id],
                opacity=0.12
            ),
            hoverinfo="none",
            showlegend=False
        )
    )

fig.add_trace(
    go.Scatter3d(
        x=flow_x,
        y=flow_y,
        z=flow_z,
        mode="lines",
        line=dict(
            color="rgba(0,255,255,0.45)",
            width=4
        ),
        hoverinfo="none",
        name="trajectory flow"
    )
)

fig.add_trace(
    go.Scatter3d(
        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],
        mode="markers",
        marker=dict(
            size=7 + 28 * (df_torus["duration"] / df_torus["duration"].max()),
            color=df_torus["time_index"],
            colorscale="Turbo",
            opacity=0.96
        ),
        text=df_torus["motif_label"],
        hoverinfo="text",
        name="vocal states"
    )
)

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(
    go.Scatter3d(
        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],
        mode="markers",
        marker=dict(
            size=12,
            color="red",
            symbol="diamond",
            line=dict(color="white", width=1)
        ),
        name="novel events"
    )
)

fig.update_layout(
    title="BLUE TIT SONG // HOLOGRAPHIC VOCAL TOPOLOGY",
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
    height=1000,
    scene=dict(
        bgcolor="black",
        xaxis=dict(gridcolor="rgba(0,255,255,0.12)"),
        yaxis=dict(gridcolor="rgba(0,255,255,0.12)"),
        zaxis=dict(gridcolor="rgba(0,255,255,0.12)"),
        camera=dict(eye=dict(x=1.7, y=1.7, z=1.2))
    )
)

fig.show()

# Directional Flow Dynamics

To model vocal state transitions as directional movement through latent acoustic space, trajectory links are converted into vector-like flow dynamics.

This representation emphasizes:
- directional acoustic progression,
- cyclic circulation behavior,
- motif transition momentum,
- and state-space navigation.

Rather than treating vocalizations as isolated syllables, the system models song production as continuous movement through an evolving dynamical manifold.

In [ ]:
# ============================================================
# DIRECTIONAL FLOW FIELD
# ============================================================

fig = go.Figure()

# ------------------------------------------------------------
# FLOW LINES
# ------------------------------------------------------------

for i in range(len(df_torus) - 1):

    x0 = df_torus.iloc[i]["x_torus"]
    y0 = df_torus.iloc[i]["y_torus"]
    z0 = df_torus.iloc[i]["z_torus"]

    x1 = df_torus.iloc[i + 1]["x_torus"]
    y1 = df_torus.iloc[i + 1]["y_torus"]
    z1 = df_torus.iloc[i + 1]["z_torus"]

    fig.add_trace(
        go.Scatter3d(
            x=[x0, x1],
            y=[y0, y1],
            z=[z0, z1],
            mode="lines",
            line=dict(
                color="rgba(0,255,255,0.65)",
                width=5
            ),
            hoverinfo="none",
            showlegend=False
        )
    )

# ------------------------------------------------------------
# FLOW NODES
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],
        mode="markers",
        marker=dict(
            size=7 + 26 * (
                df_torus["duration"] /
                df_torus["duration"].max()
            ),
            color=df_torus["time_index"],
            colorscale="Turbo",
            opacity=0.96
        ),
        text=df_torus["motif_label"],
        hoverinfo="text",
        name="vocal states"
    )
)

# ------------------------------------------------------------
# NOVEL EVENTS
# ------------------------------------------------------------

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(
    go.Scatter3d(
        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],
        mode="markers",
        marker=dict(
            size=13,
            color="red",
            symbol="diamond",
            line=dict(color="white", width=1)
        ),
        name="novel events"
    )
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig.update_layout(
    title="BLUE TIT SONG // DIRECTIONAL FLOW DYNAMICS",
    template="plotly_dark",

    paper_bgcolor="black",
    plot_bgcolor="black",

    height=1000,

    scene=dict(

        bgcolor="black",

        xaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        yaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        zaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2)
        )
    )
)

fig.show()

# Acoustic Energy Field Dynamics

To model vocal intensity as dynamical movement energy, trajectory states are scaled by latent velocity magnitude.

Large energetic nodes represent:
- rapid acoustic transitions,
- high-energy motif changes,
- and accelerated vocal movement through latent space.

Smaller nodes represent:
- stable attractor occupation,
- slower motif circulation,
- or persistent acoustic equilibrium regions.

This transforms the visualization from static geometry into an energetic state-space flow model.

In [ ]:
# ============================================================
# ENERGETIC FLOW FIELD
# ============================================================

fig = go.Figure()

# ------------------------------------------------------------
# FLOW TRAJECTORIES
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=flow_x,
        y=flow_y,
        z=flow_z,
        mode="lines",
        line=dict(
            color="rgba(0,255,255,0.45)",
            width=4
        ),
        hoverinfo="none",
        name="trajectory flow"
    )
)

# ------------------------------------------------------------
# ENERGY NODES
# ------------------------------------------------------------

energy_scale = (
    df_torus["velocity"] /
    df_torus["velocity"].max()
)

fig.add_trace(
    go.Scatter3d(
        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],

        mode="markers",

        marker=dict(
            size=6 + 42 * energy_scale,
            color=df_torus["velocity"],
            colorscale="Turbo",
            opacity=0.94,
            colorbar=dict(
                title="velocity"
            )
        ),

        text=
            "Motif: " + df_torus["motif_label"].astype(str)
            + "<br>Velocity: "
            + df_torus["velocity"].round(2).astype(str),

        hoverinfo="text",

        name="energetic states"
    )
)

# ------------------------------------------------------------
# NOVEL EVENTS
# ------------------------------------------------------------

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(
    go.Scatter3d(
        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],

        mode="markers",

        marker=dict(
            size=13,
            color="red",
            symbol="diamond",
            line=dict(color="white", width=1)
        ),

        name="novel events"
    )
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig.update_layout(
    title="BLUE TIT SONG // ENERGETIC VOCAL FLOW FIELD",

    template="plotly_dark",

    paper_bgcolor="black",
    plot_bgcolor="black",

    height=1000,

    scene=dict(

        bgcolor="black",

        xaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        yaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        zaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        camera=dict(
            eye=dict(x=1.6, y=1.5, z=1.25)
        )
    )
)

fig.show()

# Orbital Phase Synchronization

The toroidal manifold allows the estimation of cyclic orbital phase within latent acoustic space.

By projecting vocal states onto angular coordinates around the torus, recurrent synchronization structure can be explored.

This enables analysis of:
- orbital resonance,
- phase locking,
- cyclic motif recurrence,
- and anomalous phase disruptions.

The resulting representation treats birdsong as a temporally evolving oscillatory dynamical system rather than a simple sequence of discrete vocal units.

In [ ]:
# ============================================================
# ORBITAL PHASE SYNCHRONIZATION
# ============================================================

# ------------------------------------------------------------
# COMPUTE ANGULAR PHASE
# ------------------------------------------------------------

df_torus["phase_angle"] = np.arctan2(
    df_torus["y_torus"],
    df_torus["x_torus"]
)

# Normalize to 0 → 2π
df_torus["phase_angle"] = (
    df_torus["phase_angle"] + 2*np.pi
) % (2*np.pi)

# ------------------------------------------------------------
# FIGURE
# ------------------------------------------------------------

fig = go.Figure()

# ------------------------------------------------------------
# TOROIDAL FLOW
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=flow_x,
        y=flow_y,
        z=flow_z,
        mode="lines",
        line=dict(
            color="rgba(0,255,255,0.35)",
            width=4
        ),
        hoverinfo="none",
        name="orbital flow"
    )
)

# ------------------------------------------------------------
# PHASE STATES
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter3d(
        x=df_torus["x_torus"],
        y=df_torus["y_torus"],
        z=df_torus["z_torus"],

        mode="markers",

        marker=dict(
            size=7 + 28 * (
                df_torus["duration"] /
                df_torus["duration"].max()
            ),

            color=df_torus["phase_angle"],
            colorscale="HSV",
            opacity=0.96,

            colorbar=dict(
                title="orbital phase"
            )
        ),

        text=
            "Motif: "
            + df_torus["motif_label"].astype(str)
            + "<br>Phase: "
            + df_torus["phase_angle"].round(2).astype(str),

        hoverinfo="text",

        name="phase states"
    )
)

# ------------------------------------------------------------
# ANOMALIES
# ------------------------------------------------------------

novel_df = df_torus[df_torus["is_anomaly"]]

fig.add_trace(
    go.Scatter3d(
        x=novel_df["x_torus"],
        y=novel_df["y_torus"],
        z=novel_df["z_torus"],

        mode="markers",

        marker=dict(
            size=13,
            color="red",
            symbol="diamond",
            line=dict(color="white", width=1)
        ),

        name="phase anomalies"
    )
)

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------

fig.update_layout(
    title="BLUE TIT SONG // ORBITAL PHASE SYNCHRONIZATION",

    template="plotly_dark",

    paper_bgcolor="black",
    plot_bgcolor="black",

    height=1000,

    scene=dict(

        bgcolor="black",

        xaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        yaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        zaxis=dict(
            gridcolor="rgba(0,255,255,0.12)"
        ),

        camera=dict(
            eye=dict(x=1.6, y=1.5, z=1.25)
        )
    )
)

fig.show()

# Orbital Resonance Statistics

To quantify cyclic organization within the toroidal manifold, orbital phase statistics are computed for each motif state.

These measurements estimate:
- phase concentration,
- cyclic occupancy structure,
- orbital synchronization stability,
- and anomalous phase dispersion.

Highly concentrated phase distributions may indicate stable cyclic attractors, while broad distributions may reflect exploratory or transitional acoustic behavior.

In [ ]:
# ============================================================
# ORBITAL RESONANCE STATISTICS
# ============================================================

print("ORBITAL RESONANCE ANALYSIS")
print("=" * 50)

for motif in sorted(df_torus["motif_label"].unique()):

    motif_df = df_torus[
        df_torus["motif_label"] == motif
    ]

    phase_values = motif_df["phase_angle"]

    mean_phase = np.mean(phase_values)

    phase_concentration = np.std(phase_values)

    print(f"\nMotif: {motif}")

    print(
        f"Mean orbital phase: "
        f"{mean_phase:.2f}"
    )

    print(
        f"Phase dispersion: "
        f"{phase_concentration:.2f}"
    )

    if phase_concentration < 0.8:
        interpretation = (
            "high synchronization stability"
        )

    elif phase_concentration < 1.5:
        interpretation = (
            "moderate cyclic organization"
        )

    else:
        interpretation = (
            "broad exploratory phase behavior"
        )

    print(f"Interpretation: {interpretation}")

## Interpretation

The orbital resonance analysis suggests that different motif classes occupy different dynamical roles within the toroidal acoustic manifold.

The **STABLE CHIRP FIELD** motif shows moderate cyclic organization, suggesting partial recurrence within preferred orbital regions.

In contrast, the **TERMINAL DESCENT**, **TRANSITION CLOUD**, and **VERTICAL BURST** motifs exhibit broader phase dispersion, suggesting:
- exploratory circulation behavior,
- transitional acoustic movement,
- or flexible motif deployment across multiple orbital regions.

These results suggest that the vocal system may contain both:
- partially stabilized cyclic attractors,
- and dynamically exploratory transition states.

Rather than behaving as rigid symbolic units, motifs appear to participate in a continuously evolving oscillatory acoustic manifold.

# Integrated Dynamical Interpretation

The combined analyses suggest that birdsong may be modeled as movement through a structured latent dynamical manifold rather than as a purely linear sequence of discrete syllables.

Across the notebook, several interacting computational properties emerge:

- latent acoustic attractors,
- recurrent motif circulation,
- energetic transition corridors,
- orbital phase structure,
- anomaly injections,
- and partially stabilized cyclic organization.

The toroidal embedding further suggests that vocal progression may exhibit recurrent cyclic geometry rather than simple linear temporal ordering.

Within this framework:
- stable motifs behave as attractor-like orbital regions,
- transitional motifs behave as exploratory circulation states,
- and anomalous syllables behave as synchronization perturbations or state-space disruptions.

Importantly, the framework does not claim that biological vocal systems are literally toroidal.

Rather, the toroidal manifold functions as:
- an exploratory computational embedding,
- a cyclic state-space approximation,
- and a visualization architecture for studying recurrent vocal dynamics.

The notebook therefore proposes a dynamical-systems interpretation of birdsong in which vocalization is treated as:
- latent state navigation,
- oscillatory circulation,
- and probabilistic movement through acoustic topology.

# Research Directions

Potential future extensions include:

## Cross-Recording Resonance Analysis
Compare orbital structures across multiple recordings or individuals.

## Species-Level Manifold Comparison
Test whether different bird species exhibit distinct latent topological organizations.

## Neural Dynamical Correlation
Compare vocal manifold dynamics against neural recordings or motor control systems.

## Persistent Homology / Topological Data Analysis
Use TDA methods to quantify latent attractor structure.

## Dynamical Forecasting
Train predictive models on orbital trajectory flow and motif transitions.

## Self-Supervised Acoustic Embeddings
Replace PCA with neural latent representations derived from contrastive or transformer-based learning.

## Interactive Holographic Visualization
Convert the manifold into a real-time WebGL exploration environment.

## Comparative Cognitive Systems Modeling
Explore parallels between birdsong dynamics and:
- motor sequence generation,
- oscillatory neural systems,
- swarm coordination,
- or symbolic emergence in biological communication.

# Limitations, Constraints, and Future Work

## Current Constraints

This framework currently:

- analyzes only a single recording,
- does not model contextual or behavioral state,
- does not infer semantic meaning,
- and does not incorporate neural, ecological, or social interaction data.

The framework therefore should not be interpreted as demonstrating language, symbolic cognition, or semantic communication.

Rather, it explores whether birdsong may exhibit:
- structured dynamical organization,
- recurrent latent geometry,
- and partially stable acoustic state-space behavior.

---

## Methodological Limitations

Several methodological limitations remain:

- manifold geometry is sensitive to embedding choices,
- clustering outcomes are algorithm-dependent,
- transition probabilities are constrained by sample size,
- and toroidal embeddings represent computational approximations rather than biologically validated structures.

Additionally:

- PCA-based embeddings simplify high-dimensional acoustic structure,
- anomaly thresholds are heuristic,
- and orbital interpretations remain exploratory.

The present framework should therefore be interpreted as:
- computationally generative,
- hypothesis-forming,
- and exploratory rather than confirmatory.

---

## Future Research Directions

Potential future extensions include:

- cross-recording motif comparison,
- individual bird identity modeling,
- toroidal and hyperspherical embeddings,
- graph neural network sequence modeling,
- transformer-based vocal prediction,
- comparative species topology analysis,
- multimodal ecological integration,
- persistent homology analysis,
- and real-time interactive manifold exploration.

Future work could also incorporate:
- neural recordings,
- motor control measurements,
- environmental context,
- and multi-agent vocal interaction dynamics.

---

## Conceptual Direction

The long-term objective is not merely birdsong visualization.

Rather, the broader research direction investigates whether animal vocal systems can be modeled as:

- structured dynamical systems,
- latent acoustic grammars,
- oscillatory attractor networks,
- or emergent bioacoustic state-spaces.

More broadly, this work explores whether biological communication systems may exhibit:
- recurrent latent geometry,
- energetic flow organization,
- and partially stabilized topological dynamics.

The notebook therefore serves as:
- a computational exploration framework,
- a speculative systems-modeling architecture,
- and a prototype for future bioacoustic dynamical analysis.

In [ ]:
# ============================================================
# EXPORT SYLLABLE INTELLIGENCE TABLE
# ============================================================

export_cols = [
    "syllable_id",
    "start_time",
    "end_time",
    "duration",
    "cluster",
    "motif_label",
    "velocity",
    "acceleration",
    "anomaly_score",
    "is_anomaly"
]

syllable_report = syllable_df[export_cols].copy()

output_csv = "blue_tit_syllable_intelligence_report.csv"
syllable_report.to_csv(output_csv, index=False)

print("Saved:", output_csv)
syllable_report.head()

In [ ]:
# ============================================================
# FINAL BIOACOUSTIC INTELLIGENCE REPORT
# ============================================================

report = f"""
BLUE TIT BIOACOUSTIC INTELLIGENCE REPORT
========================================

Recording provenance
--------------------
Xeno-canto ID: {recording_metadata["xeno_canto_id"]}
Common name: {recording_metadata["common_name"]}
Scientific name: {recording_metadata["scientific_name"]}
Recordist: {recording_metadata["recordist"]}
Country: {recording_metadata["country"]}
Location: {recording_metadata["location"]}
Date: {recording_metadata["date"]}
Time: {recording_metadata["time"]}
Quality: {recording_metadata["quality"]}
Type: {recording_metadata["type"]}
Length: {recording_metadata["length"]}
Audio URL: {recording_metadata["audio_url"]}

Recording summary
-----------------
Species: Eurasian Blue Tit
Scientific name: Cyanistes caeruleus
Detected syllables: {len(syllable_df)}
Motif groups: {syllable_df["cluster"].nunique()}
Novel acoustic events: {syllable_df["is_anomaly"].sum()}


Dynamic state-space summary
---------------------------
Maximum velocity: {syllable_df["velocity"].max():.2f}
Maximum acceleration: {syllable_df["acceleration"].max():.2f}

Grammar summary
---------------
Most stable motif state: {most_stable_state}
Strongest cross-state transition: {strongest_transition[0]} → {strongest_transition[1]}
Dominant recurrent phrase: {top_phrase_text}
Dominant phrase occurrences: {top_phrase_count}

Interpretation
--------------
This recording shows structured movement through a latent acoustic state-space.
The bird repeatedly returns to stable motif regions while also producing
transition chains and novel acoustic events.

The dominant recurrent phrase may be treated as a candidate syntax fragment
for comparison across other recordings or individuals.
"""

print(report)

with open("blue_tit_bioacoustic_intelligence_report.txt", "w") as f:
    f.write(report)

print("Saved: blue_tit_bioacoustic_intelligence_report.txt")